# 12｜財務自由計算機 & 指標速查手冊

**這本 Notebook 是你的個人財務工具箱。**

前面 11 本談的是學術研究與市場規律；這本換你代入自己的數字，
用同樣的邏輯算出對你個人有意義的答案。

> **使用方式：** 修改每個區塊最頂端的「🔧 輸入區」數字，
> 然後執行該儲存格（Shift+Enter），圖表與結論自動更新。

## 如果明天起你不需要工作了，你需要多少錢？

大多數人的回答是「越多越好」——但這不是一個可以行動的答案。

更具體的問題是：**你的「自由生活」，每個月的實際支出是多少？**

一旦你知道這個數字，財務自由的終點線就能算出來。

以台灣生活水準為例：
- 雙北舒適生活，夫妻兩人大概需要月支出 6–8 萬元
- 用「25 倍年支出法則」（來自 Trinity Study 的 4% 提領率）
- 月支出 6 萬 × 12 個月 × 25 = **FIRE 數字：1,800 萬**

1,800 萬聽起來很遙遠——但如果你現在 28 歲，儲蓄率 40%，
每年投入報酬率 7% 的 ETF，算出來可能 22 年後就到了。

**這本 Notebook 就是你的個人財務計算機。把你的數字填進去，你的終點線就出現了。**

## 🎯 學習目標
1. 用真實數字評估自己目前的財務健康狀況
2. 計算個人的 FIRE 數字與達成時間表
3. 能用 CAPE、P/E、股息率判讀當前市場溫度

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = [
    'Microsoft YaHei', 'SimHei', 'Heiti TC',
    'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif'
]
matplotlib.rcParams['axes.unicode_minus'] = False
print('✅ 環境就緒')

---
## 一｜個人財務健檢

財務健檢不是算「你有多少錢」，而是算**五個比率**：

| 指標 | 計算方式 | 健康標準 |
|------|----------|----------|
| 儲蓄率 | 每月儲蓄 ÷ 每月稅後收入 | ≥ 20% |
| 緊急備用金月數 | 流動資產 ÷ 每月必要支出 | ≥ 6 個月 |
| 財務自由倍數 | 總資產 ÷ 年度支出 | 目標 25× |
| 負債比 | 總負債 ÷ 總資產 | ≤ 40% |
| 投資資產比 | 投資資產 ÷ 總資產 | ≥ 50%（排除自住房） |

**標準來源：** Bengen (1994) 4% 法則、一般個人理財建議（CFP 準則）

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  🔧 輸入區 — 填入你的個人財務數字（元）  ║
# ╚══════════════════════════════════════════╝

月收入_稅後   = 60_000    # 每月稅後收入
月支出_必要   = 25_000    # 每月必要生活支出（食住行）
月支出_總計   = 40_000    # 每月總支出（含娛樂、保險等）

流動資產      = 300_000   # 現金 + 活存 + 可隨時變現資產
投資資產      = 500_000   # 股票、ETF、基金、退休金帳戶
不動產        = 0         # 自住房估值（若有）

總負債        = 100_000   # 信貸、車貸、學貸（不含房貸）
房貸餘額      = 0         # 房貸（若有）

# ─── 自動計算 ────────────────────────────────────────────────
月儲蓄        = 月收入_稅後 - 月支出_總計
儲蓄率        = 月儲蓄 / 月收入_稅後 if 月收入_稅後 > 0 else 0
緊急備用金月數 = 流動資產 / 月支出_必要 if 月支出_必要 > 0 else 0
年支出         = 月支出_總計 * 12
總資產         = 流動資產 + 投資資產 + 不動產
淨資產         = 總資產 - 總負債 - 房貸餘額
財務自由倍數   = 淨資產 / 年支出 if 年支出 > 0 else 0
負債比         = (總負債 + 房貸餘額) / 總資產 if 總資產 > 0 else 0
投資資產比     = 投資資產 / (總資產 - 不動產) if (總資產 - 不動產) > 0 else 0

# ─── 評分系統 ───────────────────────────────────────────────
def score_bar(value, threshold_ok, threshold_great, reverse=False):
    """回傳 (emoji, 評語)"""
    if not reverse:
        if value >= threshold_great: return '🟢', '優秀'
        if value >= threshold_ok:    return '🟡', '達標'
        return '🔴', '需改善'
    else:  # 負債比：越低越好
        if value <= threshold_great: return '🟢', '優秀'
        if value <= threshold_ok:    return '🟡', '達標'
        return '🔴', '需改善'

metrics = [
    ('儲蓄率',         儲蓄率,         0.20,  0.30,  False, '{:.1%}',  '目標 ≥ 20%'),
    ('緊急備用金月數', 緊急備用金月數,  6,     12,    False, '{:.1f} 個月', '目標 ≥ 6 個月'),
    ('財務自由倍數',   財務自由倍數,    10,    25,    False, '{:.1f}×', '目標 25×（FIRE）'),
    ('負債比',         負債比,          0.40,  0.20,  True,  '{:.1%}',  '目標 ≤ 40%'),
    ('投資資產比',     投資資產比,      0.50,  0.70,  False, '{:.1%}',  '目標 ≥ 50%'),
]

print('=' * 52)
print('  個人財務健檢報告')
print('=' * 52)
for name, val, ok, great, rev, fmt, hint in metrics:
    emoji, rating = score_bar(val, ok, great, rev)
    print(f'  {emoji} {name:<10}  {fmt.format(val):<10}  {rating}  ({hint})')
print('=' * 52)
print(f'  月儲蓄金額：{月儲蓄:,.0f} 元')
print(f'  淨資產：    {淨資產:,.0f} 元')
print(f'  年支出：    {年支出:,.0f} 元')
print('=' * 52)

# 視覺化雷達圖
labels = ['儲蓄率\n(≥20%)', '備用金\n(≥6月)', 'FI倍數\n(25×)', '低負債\n(≤40%)', '投資比\n(≥50%)']
# 標準化到 0–1（各指標相對進度）
raw_scores = [
    min(儲蓄率 / 0.30, 1.0),
    min(緊急備用金月數 / 12, 1.0),
    min(財務自由倍數 / 25, 1.0),
    max(1 - 負債比 / 0.60, 0.0),
    min(投資資產比 / 0.80, 1.0),
]

N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
raw_scores_plot = raw_scores + [raw_scores[0]]
angles_plot     = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles_plot, raw_scores_plot, 'royalblue', linewidth=2)
ax.fill(angles_plot, raw_scores_plot, 'royalblue', alpha=0.25)
ax.plot(angles_plot, [1.0] * (N + 1), 'gray', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '目標'], fontsize=7)
ax.set_ylim(0, 1.1)
ax.set_title('個人財務健檢雷達圖\n（各項目相對目標值的達成進度）', fontsize=11, pad=20)
plt.tight_layout()
plt.show()

---
## 二｜FIRE 計算機

**FIRE = Financial Independence, Retire Early（財務獨立、提早退休）**

### FIRE 的核心邏輯（來自 Trinity Study）

1. **FIRE 數字** = 年支出 × 25（等價於 4% 安全提領率）
2. **距離 FIRE 的年數** 由三個變數決定：
   - 目前儲蓄（起點）
   - 每月儲蓄金額（每年累積速度）
   - 投資報酬率（複利加速）

### 三種 FIRE 版本

| 類型 | 定義 | 提領率 | 適合對象 |
|------|------|--------|----------|
| **Lean FIRE** | 精簡生活、低花費退休 | 5% | 能接受簡樸、無子女 |
| **Regular FIRE** | 維持現有生活水準 | 4% | 一般目標 |
| **Fat FIRE** | 退休後生活更好 | 3% | 需要更大資金池 |
| **Coast FIRE** | 現在停止額外儲蓄，讓複利跑到退休 | 4% | 年輕時達到後可轉換 |

In [ ]:
# ╔════════════════════════════════════╗
# ║  🔧 輸入區 — FIRE 個人參數設定  ║
# ╚════════════════════════════════════╝

目前年齡         = 30
希望退休年齡     = 55

目前投資資產     = 500_000     # 目前可投資的淨資產（元）
每月儲蓄         = 20_000      # 每月能存下並投入的金額
退休後年支出     = 480_000     # 退休後每年預計花費（元）

預期年報酬率     = 0.07        # 稅後實質報酬（扣除通膨，保守估計 5–7%）
通膨率           = 0.02        # 用於計算未來支出

# ─── FIRE 數字計算 ──────────────────────────────────────────
fire_regular = 退休後年支出 / 0.04   # Regular FIRE（4%）
fire_lean    = 退休後年支出 / 0.05   # Lean FIRE（5%）
fire_fat     = 退休後年支出 / 0.03   # Fat FIRE（3%）

# 每月投入
月報酬率 = (1 + 預期年報酬率) ** (1/12) - 1
年儲蓄   = 每月儲蓄 * 12

# 計算達到 FIRE 數字所需年數（FV公式反推）
def years_to_fire(target, current, monthly_saving, monthly_rate):
    if current >= target:
        return 0
    n = 0
    portfolio = current
    while portfolio < target and n < 100:
        portfolio = portfolio * (1 + monthly_rate) + monthly_saving
        n += 1/12
    return round(n, 1)

yrs_regular = years_to_fire(fire_regular, 目前投資資產, 每月儲蓄, 月報酬率)
yrs_lean    = years_to_fire(fire_lean,    目前投資資產, 每月儲蓄, 月報酬率)
yrs_fat     = years_to_fire(fire_fat,     目前投資資產, 每月儲蓄, 月報酬率)

fire_age_regular = 目前年齡 + yrs_regular
fire_age_lean    = 目前年齡 + yrs_lean
fire_age_fat     = 目前年齡 + yrs_fat

# Coast FIRE：現在停止儲蓄，讓複利跑到退休年齡
# Coast FIRE 數字 = FIRE 數字 / (1+r)^(退休年齡-現在年齡)
coast_years  = 希望退休年齡 - 目前年齡
coast_fire   = fire_regular / (1 + 預期年報酬率) ** coast_years
coast_achieved = 目前投資資產 >= coast_fire

# ─── 資產成長模擬 ────────────────────────────────────────────
max_years = max(int(yrs_regular) + 5, coast_years + 5, 40)
timeline  = np.arange(0, max_years + 1)
portfolio_growth = np.zeros(len(timeline))
portfolio_growth[0] = 目前投資資產
for i in range(1, len(timeline)):
    portfolio_growth[i] = portfolio_growth[i-1] * (1 + 預期年報酬率) + 年儲蓄

# ─── 輸出摘要 ────────────────────────────────────────────────
print('=' * 55)
print('  FIRE 計算結果')
print('=' * 55)
print(f'  目前資產：{目前投資資產:>12,.0f} 元')
print(f'  每月儲蓄：{每月儲蓄:>12,.0f} 元')
print(f'  預期年報酬：{預期年報酬率*100:.1f}%')
print()
print(f'  {'類型':<12} {'FIRE數字':>12}  {'需要年數':>8}  {'退休年齡':>8}')
print(f'  {'-'*48}')
print(f'  {'Lean FIRE (5%)':<12} {fire_lean:>12,.0f}  {yrs_lean:>7.1f}年  {fire_age_lean:>7.1f}歲')
print(f'  {'Regular FIRE (4%)':<12} {fire_regular:>12,.0f}  {yrs_regular:>7.1f}年  {fire_age_regular:>7.1f}歲')
print(f'  {'Fat FIRE (3%)':<12} {fire_fat:>12,.0f}  {yrs_fat:>7.1f}年  {fire_age_fat:>7.1f}歲')
print()
status = '✅ 已達到！現在可以停止主動儲蓄' if coast_achieved else f'❌ 尚差 {coast_fire - 目前投資資產:,.0f} 元'
print(f'  Coast FIRE 數字：{coast_fire:,.0f} 元')
print(f'  Coast FIRE 狀態：{status}')
print('=' * 55)

# ─── 視覺化 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左：資產成長軌跡
ages = 目前年齡 + timeline
axes[0].plot(ages, portfolio_growth / 1e6, color='royalblue', linewidth=2)
axes[0].fill_between(ages, 0, portfolio_growth / 1e6, alpha=0.1, color='royalblue')

for fire_val, fire_age, label, color in [
    (fire_lean,    fire_age_lean,    f'Lean FIRE\n({fire_age_lean:.0f}歲)',    'gold'),
    (fire_regular, fire_age_regular, f'Regular FIRE\n({fire_age_regular:.0f}歲)', 'royalblue'),
    (fire_fat,     fire_age_fat,     f'Fat FIRE\n({fire_age_fat:.0f}歲)',      'tomato'),
]:
    axes[0].axhline(fire_val / 1e6, color=color, linewidth=1.5, linestyle='--', alpha=0.8)
    axes[0].text(目前年齡 + 0.5, fire_val / 1e6 + 0.05, label, color=color, fontsize=8)

axes[0].axvline(希望退休年齡, color='gray', linewidth=1, linestyle=':', alpha=0.7,
                label=f'希望退休年齡 {希望退休年齡} 歲')
axes[0].set_xlabel('年齡')
axes[0].set_ylabel('資產（百萬元）')
axes[0].set_title(f'資產成長軌跡（年報酬 {預期年報酬率*100:.0f}%）', fontsize=11)
axes[0].legend(fontsize=9)

# 右：三種 FIRE 版本比較
fire_types  = ['Lean FIRE\n(提領率5%)', 'Regular FIRE\n(提領率4%)', 'Fat FIRE\n(提領率3%)']
fire_vals   = [fire_lean / 1e6, fire_regular / 1e6, fire_fat / 1e6]
fire_colors = ['#ffd700', '#4169e1', '#dc143c']
bars = axes[1].bar(fire_types, fire_vals, color=fire_colors, width=0.5, edgecolor='white', alpha=0.85)
for bar, val, yr in zip(bars, fire_vals, [yrs_lean, yrs_regular, yrs_fat]):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.1f}M\n（{yr:.0f}年後）',
                 ha='center', va='bottom', fontsize=9)

# 標出目前資產水位
axes[1].axhline(目前投資資產 / 1e6, color='green', linewidth=1.5, linestyle='--',
                label=f'目前資產 {目前投資資產/1e6:.1f}M')
axes[1].set_ylabel('所需資產（百萬元）')
axes[1].set_title('三種 FIRE 目標金額比較', fontsize=11)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 三｜儲蓄率 vs 財務自由年數

這是個人理財最反直覺的圖。

**重點不是你賺多少，而是你留下多少比例。**

儲蓄率 10% → 需要工作 40 年以上；儲蓄率 50% → 只需 16 年。
這和薪水高低無關，因為儲蓄率同時決定了「你累積資產的速度」和「你需要多少資產才夠」。

> **來源：** Mr. Money Mustache 的推廣文章（基於 Bengen/Trinity 的計算框架）

In [ ]:
# ╔══════════════════════════════════════╗
# ║  🔧 輸入區 — 調整這裡看自己的位置  ║
# ╚══════════════════════════════════════╝
你的儲蓄率     = 0.33    # 你目前的儲蓄率（月儲蓄 ÷ 月收入）
你的起始資產   = 0.0     # 目前投資資產相對於年收入的倍數（例：存了 1 年收入就填 1.0）

# ─── 計算 ─────────────────────────────────────────────────────
rates   = np.linspace(0.05, 0.75, 200)
returns = [0.05, 0.07, 0.10]  # 三種報酬率情境
colors  = ['tomato', 'royalblue', 'forestgreen']

def years_to_fi(saving_rate, annual_return, start_multiple=0):
    # 以年收入為 1 標準化
    annual_expense = 1 - saving_rate
    fi_target      = annual_expense * 25       # 25× 年支出
    portfolio      = start_multiple            # 目前資產（年收入倍數）
    annual_saving  = saving_rate
    for yr in range(1, 101):
        portfolio = portfolio * (1 + annual_return) + annual_saving
        if portfolio >= fi_target:
            return yr
    return 100

fig, ax = plt.subplots(figsize=(12, 6))

for ret, color, label in zip(returns, colors, ['報酬率 5%', '報酬率 7%', '報酬率 10%']):
    yrs = [years_to_fi(r, ret, 你的起始資產) for r in rates]
    ax.plot(rates * 100, yrs, color=color, linewidth=2, label=label)

# 標記你的位置
my_yrs_mid = years_to_fi(你的儲蓄率, 0.07, 你的起始資產)
ax.axvline(你的儲蓄率 * 100, color='black', linewidth=1.5, linestyle='--', alpha=0.6)
ax.plot(你的儲蓄率 * 100, my_yrs_mid, 'k*', markersize=16,
        label=f'你的位置（儲蓄率{你的儲蓄率*100:.0f}%，7% 報酬 → {my_yrs_mid} 年）')

# 標記關鍵儲蓄率
for sr, note in [(0.10, '10%'), (0.20, '20%'), (0.50, '50%'), (0.70, '70%')]:
    yrs_note = years_to_fi(sr, 0.07, 0)
    ax.annotate(f'{yrs_note}年', xy=(sr * 100, yrs_note), xytext=(sr * 100 + 1.5, yrs_note + 2),
                fontsize=8, color='royalblue')

ax.set_xlabel('儲蓄率 (%)', fontsize=11)
ax.set_ylabel('達到財務自由所需年數', fontsize=11)
ax.set_title('儲蓄率 vs 財務自由年數\n（從零開始，4% 安全提領率）', fontsize=12)
ax.set_xlim(5, 75)
ax.set_ylim(0, 65)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# 速查表
print('\n儲蓄率速查表（7% 年報酬，從零開始）：')
print(f"  {'儲蓄率':^8} | {'需要年數':^8} | {'退休後每年支出（月收入5萬為例）':^20}")
print('  ' + '-' * 50)
for sr in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    yrs = years_to_fi(sr, 0.07)
    annual_exp = (1 - sr) * 600_000  # 月收入5萬 × 12
    print(f"  {sr*100:^8.0f}% | {yrs:^8} | {annual_exp:>12,.0f} 元/年")

---
## 四｜市場溫度計：指標判讀速查

這一節不是讓你預測市場，而是讓你**知道現在大概在哪個估值水位**——
然後根據此調整心理預期，不是調整資產配置。

### 核心原則

> **指標是溫度計，不是操盤信號。**
> CAPE 偏高不代表明天要跌，只代表未來 10 年預期報酬可能較低。
> 長期投資者的正確做法是調整「預期」，而不是「出場」。

### 四大指標判讀邏輯

| 指標 | 計算方式 | 判讀重點 | 侷限 |
|------|----------|----------|------|
| **Shiller CAPE** | 股價 ÷ 過去10年平均通膨調整EPS | 預測未來10年報酬 | 不適合短期擇時 |
| **本益比 P/E** | 股價 ÷ 當年EPS | 當下獲利能力估值 | 景氣底部EPS低，P/E會虛高 |
| **股息率** | 年股息 ÷ 股價 | 收益型判斷；與殖利率比較 | 高股息可能是股價崩跌 |
| **巴菲特指標** | 股市總市值 ÷ GDP | 整體市場高低估 | 各國差異大，不能跨國直接比較 |

In [ ]:
# ╔═══════════════════════════════════════════╗
# ║  🔧 輸入區 — 填入當前市場數字（手動查詢）  ║
# ╚═══════════════════════════════════════════╝

# 美股
sp500_cape        = 34.5    # Shiller CAPE（查詢 multpl.com）
sp500_pe          = 25.0    # 標普500 P/E（查詢 multpl.com）
sp500_dividend    = 1.3     # 標普500 股息率 %（查詢 multpl.com）
us_buffett_pct    = 195     # 美股巴菲特指標 %（查詢 currentmarketvaluation.com）

# 台股（可選填）
taiex_pe          = 18.0    # 台股加權指數 P/E（查詢 臺灣證交所）
taiex_pb          = 2.2     # 台股 P/B 比
taiex_dividend    = 3.5     # 台股殖利率 %

# ─── 美股 CAPE 判讀 ─────────────────────────────────────────
# 根據 Shiller (2000) 和後續研究，CAPE 對未來 10 年報酬的預測：
# CAPE ~10 → 預期年化 ~10–12%；CAPE ~30 → 預期年化 ~3–5%
# 粗略公式：預期年化 = 1/CAPE × 調整係數

def cape_to_expected_return(cape):
    """簡化的 CAPE → 10 年預期年化報酬估算"""
    return 1 / cape * 0.75 + 0.02   # 加通膨 2%

cape_ranges = [
    (0,  10,  '嚴重低估', '#006400', '歷史最低水位；大機會'),
    (10, 15,  '低估',     '#228B22', '便宜；長期預期報酬高'),
    (15, 20,  '合理偏低', '#90EE90', '接近歷史均值'),
    (20, 25,  '合理',     '#FFD700', '長期均值（~17）以上'),
    (25, 30,  '偏貴',     '#FFA500', '預期報酬開始下降'),
    (30, 40,  '昂貴',     '#FF4500', '1990年代末類似水位'),
    (40, 999, '極度昂貴', '#8B0000', '歷史最高水位附近'),
]

def get_cape_zone(cape):
    for lo, hi, label, color, desc in cape_ranges:
        if lo <= cape < hi:
            return label, color, desc
    return '未知', 'gray', ''

cape_label, cape_color, cape_desc = get_cape_zone(sp500_cape)
cape_exp_ret = cape_to_expected_return(sp500_cape)

# ─── 視覺化 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：CAPE 量表
ax = axes[0]
cape_zone_bounds = [0, 10, 15, 20, 25, 30, 40, 55]
cape_zone_colors = ['#006400', '#228B22', '#90EE90', '#FFD700', '#FFA500', '#FF4500', '#8B0000']
zone_labels      = ['嚴重\n低估', '低估', '合理\n偏低', '合理', '偏貴', '昂貴', '極度\n昂貴']

for i in range(len(cape_zone_colors)):
    ax.barh(0, cape_zone_bounds[i+1] - cape_zone_bounds[i],
            left=cape_zone_bounds[i], height=0.4,
            color=cape_zone_colors[i], alpha=0.7)
    mid = (cape_zone_bounds[i] + cape_zone_bounds[i+1]) / 2
    ax.text(mid, -0.28, zone_labels[i], ha='center', va='top', fontsize=8)

ax.axvline(sp500_cape, color='black', linewidth=2.5, linestyle='-')
ax.text(sp500_cape, 0.25, f'現在\n{sp500_cape}', ha='center', fontsize=10, fontweight='bold')

# 歷史均值線
for val, label in [(17, '歷史均值\n17'), (30, '2000年\n高點')]:
    ax.axvline(val, color='gray', linewidth=1, linestyle='--', alpha=0.6)
    ax.text(val, -0.25, label, ha='center', fontsize=7, color='gray')

ax.set_xlim(5, 55)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel('Shiller CAPE 數值')
ax.set_title(f'美股 CAPE 量表\n現在：{sp500_cape} → {cape_label}\n10 年預期年化報酬：{cape_exp_ret*100:.1f}%',
             fontsize=10)

# 右：台股 vs 美股指標對比
ax2 = axes[1]
categories = ['P/E 比\n（合理區間12–20）', '股息率 %\n（越高相對較便宜）']
us_vals    = [sp500_pe,       sp500_dividend]
tw_vals    = [taiex_pe,       taiex_dividend]
pe_refs    = [(12, 20),       (1.5, 4)]

x = np.arange(len(categories))
width = 0.3
b1 = ax2.bar(x - width/2, us_vals, width, label='美股 S&P500', color='steelblue', alpha=0.8)
b2 = ax2.bar(x + width/2, tw_vals, width, label='台股加權指數', color='tomato', alpha=0.8)

for bar, val in zip(list(b1) + list(b2), us_vals + tw_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.2,
             f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 合理區間參考線（僅 P/E）
ax2.axhline(15, color='gray', linewidth=1, linestyle='--', alpha=0.5, label='P/E 長期均值 ~15')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=9)
ax2.set_title('台股 vs 美股：主要估值指標', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ─── 文字判讀 ────────────────────────────────────────────────
print(f'\n{"="*52}')
print('  市場溫度計 — 文字判讀')
print(f'  {"="*50}')

print(f'\n  【美股 S&P500】')
print(f'  Shiller CAPE  : {sp500_cape}  → {cape_label} / {cape_desc}')
print(f'  10年預期年化  : {cape_exp_ret*100:.1f}%（扣通膨後的實質報酬估算）')
print(f'  本益比 P/E    : {sp500_pe}')
pe_zone = '偏高' if sp500_pe > 22 else ('合理' if sp500_pe > 15 else '偏低')
print(f'                   → {pe_zone}（歷史均值約 15–17）')
print(f'  股息率        : {sp500_dividend}%')
dy_note = '低於10年公債殖利率，股息吸引力弱' if sp500_dividend < 4.5 else '高於公債，股息有吸引力'
print(f'                   → {dy_note}')
print(f'  巴菲特指標    : {us_buffett_pct}%')
buff_zone = '嚴重高估' if us_buffett_pct > 150 else ('偏高' if us_buffett_pct > 100 else '合理')
print(f'                   → {buff_zone}（100% 以下為合理，歷史均值約 80%）')

print(f'\n  【台股加權指數】')
print(f'  本益比 P/E    : {taiex_pe}')
tw_pe_zone = '偏高' if taiex_pe > 20 else ('合理' if taiex_pe > 13 else '偏低')
print(f'                   → {tw_pe_zone}（台股歷史均值約 15）')
print(f'  股價淨值比 PB : {taiex_pb}')
print(f'  殖利率        : {taiex_dividend}%')
print(f'                   → {"高殖利率，相對美股有優勢" if taiex_dividend > sp500_dividend else "低於近期歷史均值"}')
print(f'\n  ⚠️  以上數字需手動更新——查詢網址請見下方備注')
print('=' * 52)

print('\n📌 查詢來源：')
print('  美股 CAPE / P/E / 股息率  : https://multpl.com')
print('  巴菲特指標                : https://currentmarketvaluation.com')
print('  台股 P/E / PB / 殖利率   : https://www.twse.com.tw → 統計資料 → 市場統計')

---
## 五｜指標判讀速查手冊

### Shiller CAPE 預期報酬對應表

| CAPE 區間 | 歷史對應 | 未來 10 年預期年化報酬（含通膨） | 判讀建議 |
|-----------|----------|-----------------------------------|----------|
| < 10 | 1920s、1982年市場底部 | 12–15% | 千載難逢的買入機會 |
| 10–15 | 1970s、2009年金融危機底部 | 9–12% | 明顯偏低，長線買進 |
| 15–20 | 歷史均值（約17） | 7–9% | 合理，正常投入 |
| 20–25 | 多數正常牛市 | 5–7% | 稍貴，不急進 |
| 25–35 | 2015–2019、2021–今 | 2–5% | 偏貴，調低預期 |
| > 35 | 2000年科技泡沫（44）、2021年底（38） | 0–2%（甚至負） | 昂貴，未來報酬可能很低 |

### 本益比 P/E 區間

| P/E 範圍 | 歷史對應情境 | 說明 |
|----------|-------------|------|
| < 10 | 景氣衰退底部 | 通常是恐慌或盈利暴增時 |
| 10–15 | 熊市尾端 | 歷史上長期買點 |
| 15–20 | 正常牛市 | 合理估值 |
| 20–25 | 樂觀牛市 | 稍貴但有基本面支撐 |
| > 25 | 泡沫擴張 | 需要更高盈利增長才能撐住 |

> ⚠️ **P/E 的陷阱：** 景氣底部時 EPS 低，P/E 會虛高（看起來很貴但其實不是），此時應改用 CAPE 或 P/B 判斷。

### 股息率解讀

| 股息率 | 解讀 |
|--------|------|
| 遠低於公債殖利率 | 股市相對債市缺乏股息吸引力（如美股目前） |
| 接近公債殖利率 | 估值中性 |
| 明顯高於公債 | 股市相對便宜，或市場恐慌 |
| 異常高（>6%） | 可能是股價重挫，需確認配息能力是否可持續 |

### 巴菲特指標（股市總市值 ÷ GDP）

| 數值 | 解讀 |
|------|------|
| < 75% | 明顯低估 |
| 75–90% | 合理 |
| 90–115% | 稍貴 |
| > 115% | 高估 |
| > 150% | 巴菲特本人說「玩火」（2000年、2021年說過） |

---
## 六｜這跟你有什麼關係？

### 財務健檢的正確使用方式

這些指標不是要你做交易，而是幫你回答三個問題：

1. **「我現在財務狀況如何？」** → 用第一節的雷達圖找最弱的一項，集中改善
2. **「我距離財務自由還有多遠？」** → FIRE 計算機給你一個具體時間表，用儲蓄率換時間
3. **「市場現在貴不貴？」** → CAPE 告訴你「未來 10 年可能的報酬範圍」，調整預期，而不是擇時進出

### 三個帶走的結論

**結論 1：儲蓄率比報酬率更重要（在累積期）**
> 儲蓄率從 10% 提高到 30%，財務自由年數縮短超過 15 年。
> 同樣的資金、報酬率從 7% 提高到 10%，只縮短 3–5 年。

**結論 2：4% 法則是起點，不是終點**
> CAPE 偏高的年代開始退休，4% 可能需要降到 3.5%。
> 退休後有彈性降低花費的人，可以維持 4% 甚至更高。

**結論 3：市場溫度計影響預期，不影響策略**
> CAPE 35 ≠ 明天要跌，只代表「未來 10 年年化報酬可能是 3–5% 而非 7–9%」。
> 你的資產配置、儲蓄計畫不應該因此停擺；應調整的是你的退休資金目標規模。

## 📌 本章重點摘要
| 工具 | 用途 |
|------|------|
| 財務雷達圖 | 一眼看出五大財務指標中最需要改善的項目 |
| FIRE 計算機 | 代入個人數字，算出 Lean / Regular / Fat / Coast FIRE 時間表 |
| 儲蓄率曲線 | 理解儲蓄率對財務自由年數的非線性影響 |
| CAPE 量表 | 估算當前市場的 10 年預期年化報酬範圍 |
| P/E / 股息率 / 巴菲特指標 | 快速判斷市場相對估值水位 |
| 核心心法 | 指標是溫度計，不是操盤信號；改善儲蓄率比選時機更有效 |

> **課程完結：** 恭喜你完成實證金融學全系列課程！
> 帶走的不只是圖表，而是用數據思考財務決策的習慣。